In [1]:
import pandas as pd                                  # pandas: tables
from scipy import stats                              # scipy.stats: the t-test
from statsmodels.stats.power import TTestIndPower    # sample-size calculator for number outcomes

In [2]:
url = "https://raw.githubusercontent.com/assemmkhalil/MarketingABtest/main/Marketing-Campaign.csv"
df = pd.read_csv(url)        # all 3 promotions this time: 548 rows (137 restaurants x 4 weeks)

restaurants = df.groupby(["LocationID", "Promotion", "MarketSize"])["SalesInThousands"].mean().reset_index()
print(restaurants.shape)     # 137 rows: one per restaurant

(137, 4)


### The question, the hypotheses and the stricter line
Question: do the three promotions give different average weekly sales? <br>
Two at a time gives three pairs: 1 vs 2, 1 vs 3, 3 vs 2. For each pair: <br>
• Null hypothesis: the two promotions give the same average sales. <br>
• Alternative hypothesis: they give different average sales. <br>
Three tests, so the stricter line: 0.05 ÷ 3 = 0.0167.

Was the sample big enough?

spread 16, smallest change worth catching 10, effect size 0.625, power 80%

In [4]:
needed = TTestIndPower().solve_power(effect_size=10/16, alpha=0.05 / 3, power=0.8)
print(needed)    # restaurants needed for each promotion

55.05601498767832


It prints 55.1, so 56 restaurants per promotion. The test had only 43, 47 and 47. The test was a bit too small for three groups: the stricter line needs more restaurants than Project B did (42). Its power is about 70% instead of 80%, so a real difference of 10 thousand dollars could be missed more often.

In [5]:
counts = restaurants["Promotion"].value_counts().sort_index()   # restaurants per promotion, in order 1, 2, 3
print(counts)
print((counts / counts.sum() * 100).round(1))                    # each group as a percent
print(pd.crosstab(restaurants["Promotion"], restaurants["MarketSize"]))   # area size in each group

Promotion
1    43
2    47
3    47
Name: count, dtype: int64
Promotion
1    31.4
2    34.3
3    34.3
Name: count, dtype: float64
MarketSize  Large  Medium  Small
Promotion                       
1              14      24      5
2              16      27      4
3              12      29      6


In [6]:
print(stats.chisquare(counts).pvalue)

0.8897735194498095


Above 0.05, so the 43/47/47 split is normal chance.

Novelty effect???

In [7]:
weekly = pd.crosstab(df["week"], df["Promotion"], values=df["SalesInThousands"], aggfunc="mean")   # average sales per week and promotion
print(weekly.round(1))

Promotion     1     2     3
week                       
1          58.2  47.7  55.8
2          56.9  47.6  55.9
3          58.8  47.7  54.4
4          58.4  46.3  55.4


Stable over the 4 weeks.

In [8]:
promo_1 = restaurants.loc[restaurants["Promotion"] == 1, "SalesInThousands"]   # 43 restaurant averages
promo_2 = restaurants.loc[restaurants["Promotion"] == 2, "SalesInThousands"]   # 47 restaurant averages
promo_3 = restaurants.loc[restaurants["Promotion"] == 3, "SalesInThousands"]   # 47 restaurant averages

print(round(promo_1.mean(), 1), round(promo_2.mean(), 1), round(promo_3.mean(), 1))   # average of each promotion

print("1 vs 2:", stats.ttest_ind(promo_1, promo_2).pvalue)
print("1 vs 3:", stats.ttest_ind(promo_1, promo_3).pvalue)
print("3 vs 2:", stats.ttest_ind(promo_3, promo_2).pvalue)

58.1 47.3 55.4
1 vs 2: 0.0012025674613766499
1 vs 3: 0.4263564328199861
3 vs 2: 0.013531461150953565


## What these results mean:
• 1 vs 2: p = 0.0012 is below 0.0167, so the difference is real. Promotion 1 sells about 10.8 thousand dollars more per restaurant per week than promotion 2 (58.1 − 47.3). <br>
• 3 vs 2: p = 0.0135 is below 0.0167, so the difference is real. Promotion 3 sells about 8.0 thousand dollars more per restaurant per week than promotion 2 (55.4 − 47.3). <br>
• 1 vs 3: p = 0.43 is above 0.0167, so the null hypothesis stays: no real difference between promotions 1 and 3. Their gap (58.1 − 55.4 = 2.7) is just chance. <br>
<br>
<b>In one sentence: promotion 2 is clearly the weakest; promotions 1 and 3 are equally good.

How big are the effects? (Cohen's d)

In [9]:
gap = promo_1.mean() - promo_2.mean()                              # 58.1 - 47.3
combined_sd = ((promo_1.std()**2 + promo_2.std()**2) / 2) ** 0.5    # typical spread of the two groups together
print(gap / combined_sd)                                            # Cohen's d for 1 vs 2

0.7047313069495937


### What the effect sizes mean:
• 1 vs 2 (d = 0.70): medium to large. The gap is about 0.7 times the normal difference between restaurants: a clear gap that a manager would notice in the sales reports. <br>
• 3 vs 2 (d = 0.52): medium. Also a clear gap. <br>
• 1 vs 3: no effect size, because the difference is not real (p = 0.43).

In one sentence: promotions 1 and 3 are both clearly better than promotion 2. The two numbers 0.70 and 0.52 look different, but we cannot say promotion 1 beats promotion 3, because the test of 1 against 3 showed their difference is just chance.

## Decision
Report sentence: "Promotion 1 sold on average 58.1 thousand dollars per restaurant per week, promotion 3 sold 55.4 and promotion 2 sold 47.3. Promotions 1 and 3 both sold clearly more than promotion 2 (t-test p = 0.0012 and p = 0.0135, both below the corrected line of 0.0167). There was no clear difference between promotions 1 and 3 (p = 0.43)."


### Decision:
• Stop using promotion 2. <br>
• Between promotions 1 and 3, the data cannot pick a winner. Choose the cheaper one. For a clear answer, run a new test of 1 against 3 only; their difference is small (2.7), so that test will need many more restaurants.